In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import numpy as np
print(np.__version__)


2.0.2


In [ ]:
!bash /content/drive/MyDrive/V2_drift/texturemodel/code/run/extract_texture_feats_second.sh

allexport      	off
braceexpand    	on
emacs          	off
errexit        	on
errtrace       	off
functrace      	off
hashall        	on
histexpand     	off
history        	off
ignoreeof      	off
interactive-comments	on
keyword        	off
monitor        	off
noclobber      	off
noexec         	off
noglob         	off
nolog          	off
notify         	off
nounset        	on
onecmd         	off
physical       	off
pipefail       	off
posix          	off
privileged     	off
verbose        	off
vi             	off
xtrace         	off
/usr/bin/python3
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
PYTHONPATH: /content/drive/MyDrive/V2_drift/texturemodel:/content/drive/MyDrive/V2_drift/texturemodel/code:/content/drive/MyDrive/V2_drift/texturemodel/code/utils:/env/python
numpy: 2.0.2
pyrtools: 1.0.10
PWD

In [ ]:
drive.flush_and_unmount()

In [ ]:
import time, os
print("alive:", time.time())


In [ ]:
import os
import numpy as np
import h5py

fn = "/content/drive/MyDrive/V2_drift/features/pyramid_texture/S3_features_each_prf_4ori_4sf_grid5_npix256_first.h5py"

assert os.path.exists(fn), f"File not found: {fn}"
print("File:", fn)
print("Size (GB):", os.path.getsize(fn) / (1024**3))

rng = np.random.default_rng(0)

# --- how much to test ---
n_images_for_entry_checks = 500   # number of images to sample
entries_per_image = 400          # random (feature,prf) pairs per sampled image
n_rows_full_check = 20           # full-row min/max/anyNaN checks (heavier but safe)

with h5py.File(fn, "r") as f:
    ds = f["features"]
    n_images, n_features, n_prfs = ds.shape

    print("\nDataset:", ds.name)
    print("  shape:", ds.shape)
    print("  dtype:", ds.dtype)

    # ------------------------------------------------------------
    # Check A: random-entry NaN/Inf + non-trivial distribution
    # (h5py-safe: sort img indices, then read per-image vectors)
    # ------------------------------------------------------------
    img_idx = rng.choice(n_images, size=n_images_for_entry_checks, replace=False)
    img_idx.sort()  # IMPORTANT for h5py

    nan_cnt = 0
    inf_cnt = 0
    zero_cnt = 0
    total_cnt = 0

    global_min = np.inf
    global_max = -np.inf

    for i in img_idx:
        f_idx = rng.integers(0, n_features, size=entries_per_image, endpoint=False)
        p_idx = rng.integers(0, n_prfs,     size=entries_per_image, endpoint=False)

        # pull one full image row (641x1456 ~ 3.7MB) and sample it
        # (this is the simplest reliable way with h5py)
        row = ds[i, :, :]  # numpy array
        vals = row[f_idx, p_idx]

        nan_cnt += np.isnan(vals).sum()
        inf_cnt += np.isinf(vals).sum()
        zero_cnt += (vals == 0).sum()
        total_cnt += vals.size

        vmin = float(np.min(vals))
        vmax = float(np.max(vals))
        global_min = min(global_min, vmin)
        global_max = max(global_max, vmax)

    print("\n[Check A] Random-entry checks")
    print(f"  sampled images: {len(img_idx)}")
    print(f"  entries per image: {entries_per_image}")
    print(f"  total sampled entries: {total_cnt}")
    print(f"  NaNs: {nan_cnt}")
    print(f"  Infs: {inf_cnt}")
    print(f"  zeros: {zero_cnt} ({100*zero_cnt/total_cnt:.2f}%)")
    print(f"  sampled min/max: {global_min:.6g} / {global_max:.6g}")

    assert nan_cnt == 0 and inf_cnt == 0, "Found NaNs/Infs in sampled entries!"

    # ------------------------------------------------------------
    # Check B: verify second half is not all zeros
    # (the exact bug you saw before)
    # ------------------------------------------------------------
    def full_row_stats(idx):
        row = ds[idx, :, :]
        return {
            "idx": idx,
            "min": float(np.min(row)),
            "max": float(np.max(row)),
            "mean": float(np.mean(row)),
            "std": float(np.std(row)),
            "any_nan": bool(np.isnan(row).any()),
            "all_zero": bool(np.all(row == 0)),
        }

    print("\n[Check B] Full-row spot checks")
    spots = [0, 1, 10, 100, 4999, 5000, 5001, 7500, 9999]
    spots = [s for s in spots if 0 <= s < n_images]

    for s in spots:
        st = full_row_stats(s)
        print(f"  img {st['idx']:>4}: min={st['min']:.6g} max={st['max']:.6g} "
              f"mean={st['mean']:.6g} std={st['std']:.6g} any_nan={st['any_nan']} all_zero={st['all_zero']}")

    assert not full_row_stats(0)["all_zero"], "Image 0 row is all zeros!"
    assert not full_row_stats(5000)["all_zero"], "Image 5000 row is all zeros! (second half looks empty)"

    # ------------------------------------------------------------
    # Check C: random full-row checks (heavier)
    # ------------------------------------------------------------
    rows = rng.choice(n_images, size=n_rows_full_check, replace=False)
    print(f"\n[Check C] Random full-row checks (n={n_rows_full_check})")
    bad = []
    for r in rows:
        st = full_row_stats(int(r))
        if st["any_nan"] or st["all_zero"]:
            bad.append(st)
    if bad:
        print("  BAD rows found:")
        for st in bad:
            print(st)
        raise RuntimeError("Some rows are NaN or all zeros.")
    else:
        print("  OK: no NaNs and no all-zero rows in this random set.")

    # ------------------------------------------------------------
    # Check D: batch_done if exists
    # ------------------------------------------------------------
    if "batch_done" in f:
        done = f["batch_done"][:].astype(int)
        print("\n[Check D] batch_done:")
        print(done)
        print("  done:", done.sum(), "/", len(done))

print("\n✅ All checks passed (with h5py-safe indexing).")


File: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S3_features_each_prf_4ori_4sf_grid5_npix256_first.h5py
Size (GB): 16.688899517059326

Dataset: /features
  shape: (10000, 641, 1456)
  dtype: float32

[Check A] Random-entry checks
  sampled images: 500
  entries per image: 400
  total sampled entries: 200000
  NaNs: 0
  Infs: 0
  zeros: 114839 (57.42%)
  sampled min/max: -478.431 / 7468.54

[Check B] Full-row spot checks
  img    0: min=-1750.13 max=6120.59 mean=10.6512 std=137.289 any_nan=False all_zero=False
  img    1: min=-1426.63 max=4103.7 mean=8.42297 std=106.075 any_nan=False all_zero=False
  img   10: min=-1056.15 max=3618.38 mean=7.79195 std=101.659 any_nan=False all_zero=False
  img  100: min=-510.978 max=2159.35 mean=5.0954 std=65.1573 any_nan=False all_zero=False
  img 4999: min=0 max=0 mean=0 std=0 any_nan=False all_zero=True
  img 5000: min=0 max=0 mean=0 std=0 any_nan=False all_zero=True
  img 5001: min=0 max=0 mean=0 std=0 any_nan=False all_zero=True
  img

AssertionError: Image 5000 row is all zeros! (second half looks empty)